In [2]:

import pandas as pd
from wearable_affect.data import PROJECT_ROOT
from wearable_affect.evaluation import loso_evaluate, summarise
from wearable_affect.models import make_dummy, make_logreg, make_lightgbm

features = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "features.parquet")
print(features.shape)
print(features.groupby("target")[["scl_mean", "scr_count", "hr_mean"]].mean())
print(features[["hr_mean", "hrv_sdnn", "hrv_rmssd"]].describe().T)

(1015, 17)
        scl_mean  scr_count    hr_mean
target                                
0       1.339609   7.249649  75.696366
1       3.439927  19.870861  88.704947
            count        mean         std        min         25%         50%  \
hr_mean    1015.0   79.566900   11.227760  55.631848   72.510533   78.046512   
hrv_sdnn   1015.0  273.550038  298.530682  36.681316  168.113387  223.704555   
hrv_rmssd  1015.0  365.273441  409.705654  28.894648  227.896178  302.398034   

                  75%          max  
hr_mean     84.918366   140.909141  
hrv_sdnn   285.927626  4923.734937  
hrv_rmssd  381.698599  7186.770760  


In [2]:
results = {}
for name, make_model in [("dummy", make_dummy), ("logreg", make_logreg), ("lightgbm", make_lightgbm)]:
    results[name] = loso_evaluate(features, make_model)
    print(f"\n{name}")
    print(summarise(results[name]).round(3))


dummy
                    mean    std
macro_f1           0.413  0.004
balanced_accuracy  0.500  0.000
auroc              0.500  0.000

logreg
                    mean    std
macro_f1           0.831  0.112
balanced_accuracy  0.852  0.097
auroc              0.966  0.047

lightgbm
                    mean    std
macro_f1           0.850  0.144
balanced_accuracy  0.866  0.120
auroc              0.976  0.038


In [3]:
per_subject = pd.DataFrame({name: r.set_index("subject_id")["macro_f1"] for name, r in results.items()})
per_subject.round(3)

,dummy,logreg,lightgbm
subject_id,,,
S10,0.405,0.891,0.736
S11,0.412,0.966,0.890
S13,0.415,0.636,1.000
S14,0.415,0.894,0.415
S15,0.410,0.650,0.930
S16,0.412,0.811,0.982
S17,0.405,0.902,0.820
S2,0.414,0.678,0.852
S3,0.412,0.791,0.735


In [3]:
from wearable_affect.evaluation import loso_predict, score_predictions

preds = {name: loso_predict(features, f) for name, f in
         [("logreg", make_logreg), ("lightgbm", make_lightgbm)]}
scores = {name: score_predictions(p) for name, p in preds.items()}

auroc = pd.DataFrame({name: s.set_index("subject_id")["auroc"] for name, s in scores.items()})
auroc.round(3)

,logreg,lightgbm
subject_id,,
S2,0.965,0.994
S3,0.912,0.884
S4,0.999,1.000
S5,0.976,0.984
S6,0.987,0.922
S7,0.956,1.000
S8,1.000,1.000
S9,0.868,1.000
S10,0.999,1.000


In [5]:
s14 = preds["lightgbm"].query("subject_id == 'S14'")
s14.groupby("target")["prob"].describe()

,count,mean,std,min,25%,50%,75%,max
target,,,,,,,,
0,49.0,0.000073,0.000109,0.000026,0.000031,0.000048,0.000078,0.000776
1,20.0,0.043647,0.060083,0.000046,0.000101,0.001094,0.092624,0.204752


In [6]:
features.groupby(["subject_id", "target"])["scl_mean"].mean().unstack()

target,0,1
subject_id,,
S10,0.406831,2.539428
S11,3.036266,3.416448
S13,5.311550,14.200770
S14,0.292863,0.336285
S15,0.354735,1.342070
S16,0.320426,2.050098
S17,1.282901,1.045031
S2,0.250323,0.860700
S3,1.390550,2.837111


In [4]:
import numpy as np
from sklearn.metrics import f1_score

rows = []
for sid, g in preds["lightgbm"].groupby("subject_id"):
    thresholds = np.linspace(0.05, 0.95, 19)
    f1s = [f1_score(g["target"], (g["prob"] >= t).astype(int), average="macro") for t in thresholds]
    best = int(np.argmax(f1s))
    rows.append({"subject_id": sid, "f1_at_0.5": f1s[9], "best_threshold": thresholds[best], "f1_best": f1s[best]})

oracle = pd.DataFrame(rows).sort_values("subject_id", key=lambda s: s.str[1:].astype(int))
print(oracle.round(3))
print(oracle[["f1_at_0.5", "f1_best"]].mean().round(3))

   subject_id  f1_at_0.5  best_threshold  f1_best
7          S2      0.852            0.05    0.876
8          S3      0.735            0.85    0.807
9          S4      0.962            0.05    0.981
10         S5      0.944            0.20    0.982
11         S6      0.826            0.90    0.880
12         S7      0.868            0.95    0.931
13         S8      0.907            0.05    0.946
14         S9      0.876            0.05    0.876
0         S10      0.736            0.95    0.906
1         S11      0.890            0.90    0.920
2         S13      1.000            0.35    1.000
3         S14      0.415            0.05    0.731
4         S15      0.930            0.10    0.948
5         S16      0.982            0.05    0.982
6         S17      0.820            0.85    0.891
f1_at_0.5    0.85
f1_best      0.91
dtype: float64
